# 🌊 03 · Waves and frequencies: the Fourier transform for biologists
### Resolution, filters, blurring, finding shifts, and why fast alignment and spectral pooling work in Fourier space

**BiologyToResearch** · Part A — Foundations · Module 03 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/03_fourier_frequency.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy (`np.fft`), SciPy (`scipy.fft.dctn`, `ndimage`), Matplotlib |
| **Prerequisites** | [Module 01](01_python_arrays_density_maps.ipynb) (arrays, maps, voxel size) and [Module 02](02_rotations_geometry.ipynb) (rotations). Trigonometry: sin and cos. |
| **Estimated time** | 120–150 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | Foundations for Xu, Beck & Alber 2012 (Fourier magnitudes separate rotation from translation) and Gum-Net 2020 (DCT spectral pooling) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All
> experiments in this course are small, controlled *toy versions* of published methods. They teach the core idea of
> each paper; they do **not** reproduce the papers' numbers. **Nothing is stored in the repository:** molecules —
> including β-galactosidase (PDB 5A1A) — are real Protein Data Bank (PDB) models, downloaded the first time a notebook
> needs them and cached on your computer (offline, code-built stand-ins of the same size and symmetry are used
> instead); all images and data are simulated by code.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Describe** a wave by its amplitude, frequency (or wavelength) and phase, and **build** a density profile as a sum
   of waves.
2. **Compute and interpret** 1D, 2D and 3D Fourier transforms with NumPy (`fft`, `fftfreq`, `fftshift`, amplitude,
   phase, power spectrum).
3. **Convert** a spatial frequency into a resolution in Å, and **explain** the Nyquist limit and aliasing.
4. **Apply** low-, high- and band-pass filters, **explain** ringing, and **verify** the convolution theorem.
5. **Find** the shift between two noisy images, and particles in a micrograph, by FFT cross-correlation.
6. **Show** that a translation changes only the Fourier phase, and **use** Fourier magnitudes to find a rotation without
   knowing the shift — the idea behind fast subtomogram alignment (Xu, Beck & Alber 2012).
7. **Demonstrate** that phases carry most of the structure (amplitude–phase swap), and **compute** radially averaged
   spectra on Fourier shells (a preview of FSC).
8. **Implement** DCT spectral pooling and **measure** how it preserves positions better than max pooling (Gum-Net's
   design choice).

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, data | 5 min |
| 1 · Waves | amplitude, frequency, phase | 10 min |
| 2 · The Fourier transform in 1D | complex numbers as arrows; `np.fft` | 15 min |
| 3 · Building a profile from waves | how many waves do you need to resolve a membrane? | 10 min |
| 4 · 2D Fourier transforms | gratings, `fftshift`, spectra of molecules | 15 min |
| 5 · Frequency ↔ resolution | Å, Nyquist, aliasing, a high-resolution map at 30/15/8 Å | 10 min |
| 6 · Filters | low/high/band-pass, ringing | 10 min |
| 7 · Convolution theorem | blur = multiplication | 10 min |
| 8 · Cross-correlation | find a shift; find particles | 15 min |
| 9 · Translation only changes phase | the trick behind fast rotational search | 15 min |
| 10 · Amplitude vs phase | which carries the structure? | 5 min |
| 11 · 3D: Fourier shells | radial spectra, FSC and missing-wedge previews | 10 min |
| 12 · DCT and spectral pooling | Gum-Net's position-preserving downsampling | 15 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

"Our map resolves α-helices at 8 Å." "Side chains appear at 3.5 Å." "The missing wedge elongates particles." "We
low-pass filtered the tomogram to 40 Å." Structural biologists speak about their data in the language of
**frequencies**: resolution is the finest repeating detail you can trust; microscopes transmit some frequencies and
suppress others; tilting removes a whole wedge of them. And nearly every fast algorithm — template matching, alignment,
averaging, validation — runs in **Fourier space**, where shifting, blurring and comparing become simple multiplications.
This module builds that language from scratch, with sines and cosines and real molecules.

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook only needs a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed (not needed in this notebook).")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import scipy
from scipy import fft as sfft
from scipy import ndimage

SEED = 0
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
print("NumPy", np.__version__, "| SciPy", scipy.__version__, "| seed", SEED)

### 🧰 Course toolkit

Copied from [`course_toolkit.py`](../course_toolkit.py): map building, the Fourier helpers taught in this module
(`freq_grid`, `radial_frequency`, `apply_fourier_mask`, `lowpass`, `missing_wedge_mask`, `fsc`), plus `rotate_image`,
`add_noise_snr`, `normalize` and `pearson`.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v3.0; copied so this notebook runs alone) ──
import os
import struct

import numpy as np
from scipy import ndimage

ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2330, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=3330, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3320, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=710, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=910, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1280, symmetry="C1"),
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1910, symmetry="C2"),
    "1bxn": dict(name="RuBisCO (L4S4 half)", organism="Cupriavidus necator", mass_kDa=290, symmetry="C4"),
    "1qvr": dict(name="ClpB (3 of 6 subunits)", organism="Thermus thermophilus", mass_kDa=290, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=570, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=190, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=260, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=230, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=44, symmetry="C1"),
}
_EXTRA_ENTRIES = {"5a1a": dict(name="β-galactosidase", organism="E. coli", mass_kDa=465, symmetry="D2")}
_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def _rx(a):
    return np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])


def _ry(a):
    return np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])


_CHAIN_IDS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
def _ring(n, radius, z, radii, phase=0.0, tilt=None):
    """n identical subunits (semi-axes `radii` nm: radial, tangential, axial) around the z axis."""
    out = []
    for k in range(n):
        a = phase + 2 * np.pi * k / n
        R = _rz(a) if tilt is None else _rz(a) @ tilt
        out.append(dict(radii=radii, centre=(radius * np.cos(a), radius * np.sin(a), z), R=R, key=("ring", radii, tilt is None)))
    return out


def _lobe(radii, centre, R=None):
    return [dict(radii=radii, centre=centre, R=np.eye(3) if R is None else R, key=None)]


def _dihedral(subunits):
    """Add the partners generated by a 2-fold axis along x (dihedral symmetry)."""
    Rx = _rx(np.pi)
    return subunits + [dict(radii=s["radii"], centre=tuple(Rx @ np.array(s["centre"])), R=Rx @ s["R"], key=s["key"])
                       for s in subunits]


def _blueprint(pid):
    """Subunit layout (nm) of the code-generated stand-in for each zoo entry."""
    if pid in ("4v4r", "4v7r", "5mrc"):
        s = {"4v4r": 1.0, "4v7r": 1.13, "5mrc": 1.14}[pid]
        L = [_lobe((9.0 * s, 8.0 * s, 6.5 * s), (0, 0, 4.5 * s)),                       # large subunit body
             _lobe((3.0 * s, 3.0 * s, 3.0 * s), (0, 1.0 * s, 11.0 * s)),                # central protuberance
             _lobe((2.0 * s, 2.0 * s, 3.5 * s), (-9.0 * s, 0, 7.0 * s), _ry(-0.6)),     # L1-like stalk
             _lobe((1.4 * s, 1.4 * s, 4.0 * s), (9.5 * s, 0, 6.5 * s), _ry(0.7)),       # L7/L12-like stalk
             _lobe((7.5 * s, 5.0 * s, 3.8 * s), (1.0 * s, 0, -4.5 * s)),                # small subunit body
             _lobe((3.5 * s, 3.2 * s, 3.0 * s), (-5.5 * s, -1.0 * s, -6.5 * s))]        # small subunit head
        if pid == "4v7r":                                                                # eukaryotic expansion segments
            L += [_lobe((3.2, 2.6, 2.8), (8.0, 7.0, 7.0)), _lobe((3.0, 2.6, 3.2), (-6.0, -8.0, -6.5)),
                  _lobe((2.4, 2.4, 3.6), (-10.5, 4.0, -1.0), _ry(0.5))]
        if pid == "5mrc":                                                                # mitochondria-specific proteins
            L += [_lobe((4.2, 3.6, 2.8), (0, -7.5, 12.5)), _lobe((3.4, 3.0, 3.4), (9.0, -6.0, -8.0)),
                  _lobe((3.0, 3.0, 3.0), (-9.5, 7.0, 1.0)), _lobe((2.5, 4.5, 2.0), (2.0, 9.0, 9.0))]
        prng = np.random.default_rng(_seed(pid) + 1)                                     # rRNA-like surface bumps
        for _ in range(8):
            d = prng.normal(size=3); d /= np.linalg.norm(d)
            L += [_lobe(tuple(prng.uniform(1.2, 1.9, 3)), tuple(s * d * np.array([9.5, 8.0, 10.5])), _rz(prng.uniform(0, 6)))]
        return [x for l in L for x in l]
    if pid in ("3j9i", "6utj", "4cr2"):
        core = []
        for z, radii, ph in [(1.9, (1.9, 1.7, 2.0), 0.0), (5.6, (2.1, 1.7, 1.85), np.pi / 7)]:
            core += _ring(7, 3.6, z, radii, ph)
        core = _dihedral(core)                                                           # α7β7β7α7, D7
        if pid == "6utj":
            core += _ring(7, 3.3, 10.0, (1.9, 1.6, 2.4), np.pi / 14)                     # one heptameric cap: C7
        if pid == "4cr2":
            core += _ring(6, 4.3, 9.6, (2.2, 1.8, 2.0), 0.3)                             # base ring of six
            core += (_lobe((4.0, 3.0, 3.0), (3.5, 0, 14.5)) + _lobe((3.0, 2.5, 3.5), (-1.5, 3.0, 15.5))
                     + _lobe((2.5, 2.5, 2.5), (-3.5, -2.0, 13.5)) + _lobe((3.0, 3.0, 2.5), (-4.5, 0.5, 10.5)))
        return core
    if pid == "4v94":
        one = []
        for z in (3.6, -3.6):
            one += _ring(8, 5.2, z, (2.6, 2.0, 3.4), np.pi / 8 if z < 0 else 0.0)
        two = []
        for dx, rot in [(-8.5, 0.0), (8.5, np.pi)]:
            for s in one:
                c = _rz(rot) @ np.array(s["centre"])
                two.append(dict(radii=s["radii"], centre=(c[0] + dx, c[1], c[2]), R=_rz(rot) @ s["R"], key=s["key"]))
        return two
    if pid == "1bxn":
        # like the deposited file (the crystal's asymmetric unit): one L4S4 layer = half of the L8S8 enzyme, C4.
        # `_dihedral` of this list would give the complete enzyme (D4).
        return _ring(4, 3.6, 2.2, (2.6, 2.4, 2.4)) + _ring(4, 2.6, 5.0, (1.9, 1.7, 1.35), np.pi / 4)
    if pid == "1qvr":
        # like the deposited file: three neighbouring subunits (half of the C6 hexameric ring), C1
        tiers = [_ring(6, 4.2, 2.2, (2.2, 2.0, 2.0)), _ring(6, 3.9, -2.2, (2.3, 2.0, 2.0), np.pi / 6),
                 _ring(6, 7.2, 2.4, (0.9, 0.9, 4.6), 0.4, tilt=_rx(1.2)),               # coiled-coil 'propellers'
                 _ring(6, 3.8, 6.6, (1.7, 1.6, 1.8), 0.5)]                               # N-domains on top
        return [s for t in tiers for s in t[:3]]
    if pid == "3cf3":
        return (_ring(6, 4.0, 2.0, (2.1, 2.0, 2.0)) + _ring(6, 4.0, -2.2, (2.2, 2.0, 2.1), np.pi / 6)
                + _ring(6, 6.6, 3.0, (2.0, 1.6, 1.8), 0.25))                             # N-domains flush: C6
    if pid == "2cg9":
        half = (_lobe((1.9, 1.7, 1.7), (2.2, 0.8, 6.0)) + _lobe((2.2, 1.8, 2.0), (2.6, -0.4, 2.0))
                + _lobe((1.7, 1.5, 1.8), (1.4, 0.3, -2.2)) + _lobe((1.4, 1.2, 1.2), (4.2, 1.6, 7.6)))
        Rz = _rz(np.pi)
        return half + [dict(radii=h["radii"], centre=tuple(Rz @ np.array(h["centre"])), R=Rz @ h["R"], key=None) for h in half]
    if pid == "1u6g":
        out = []
        for k in range(6):                                                                # curved cullin stalk
            t = k / 5
            out += _lobe((1.4, 1.4, 1.6), (8.0 * np.sin(0.9 * t) - 3.0, 0, 9.0 * t - 4.5), _ry(0.9 * t))
        out += _lobe((2.2, 2.0, 2.0), (4.0, 0, 5.8)) + _lobe((1.2, 1.1, 1.1), (6.0, 1.8, 6.5))
        for k in range(10):                                                               # Cand1-like superhelix
            a = 1.6 * np.pi * k / 9
            out += _lobe((1.5, 1.4, 1.4), (-3.0 + 7.0 * np.sin(0.9 * k / 9) + 2.6 * np.cos(a), 2.6 * np.sin(a), 9.0 * k / 9 - 4.5))
        return out
    if pid == "3d2f":
        comp = (_lobe((1.8, 1.6, 1.8), (0, 0, 0)) + _lobe((1.8, 1.6, 1.8), (0, 3.2, 0)) + _lobe((1.6, 1.4, 2.4), (3.0, 1.6, 1.8))
                + _lobe((1.6, 1.5, 1.6), (-3.2, 1.0, -1.0)) + _lobe((1.6, 1.5, 1.6), (-3.2, 3.6, -0.6)))
        R2, t2 = _rx(2.3) @ _rz(0.7), np.array([1.5, -4.5, 5.5])
        return comp + [dict(radii=c["radii"], centre=tuple(R2 @ np.array(c["centre"]) + t2), R=R2 @ c["R"], key=None) for c in comp]
    if pid == "1s3x":
        return (_lobe((1.4, 1.4, 1.6), (-1.6, 0, -1.2)) + _lobe((1.3, 1.3, 1.5), (-1.4, 0.2, 1.6))
                + _lobe((1.4, 1.4, 1.6), (1.6, 0, -1.2)) + _lobe((1.2, 1.3, 1.4), (1.5, -0.3, 1.8)))
    raise KeyError(pid)


def _seed(pid):
    return int.from_bytes(pid.encode(), "little") % (2 ** 31)


_SCALE = {"4v4r": 0.923, "4v7r": 0.906, "5mrc": 0.883, "3j9i": 0.930, "6utj": 0.930, "4cr2": 0.930, "4v94": 0.886,
          "1bxn": 0.930, "1qvr": 0.885, "3cf3": 0.935, "2cg9": 0.995, "1u6g": 0.934, "3d2f": 0.997, "1s3x": 0.884}
def standin_atoms(pid):
    """Atomic model (dict of arrays) of the code-built stand-in for zoo entry `pid` (Å, centred at the origin).

    Used when the real entry cannot be downloaded: ideal α-helices arranged with the size, mass and symmetry of the
    PDB entry. Deterministic (the random numbers are seeded by the entry's name)."""
    pid = pid.lower()
    rng = np.random.default_rng(_seed(pid))
    built, parts = {}, []
    f = _SCALE.get(pid, 1.0)
    for k, s in enumerate(_blueprint(pid)):
        radii_A = tuple(10.0 * f * r for r in s["radii"])
        key = s["key"]
        if key is not None and key in built:
            sub = built[key]
        else:
            sub = helix_bundle(radii_A, rng)
            if key is not None:
                built[key] = sub
        part = _transform(sub, s["R"], 10.0 * f * np.array(s["centre"]))
        part["chain"] = np.full(len(part["xyz"]), _CHAIN_IDS[k % len(_CHAIN_IDS)])
        parts.append(part)
    atoms = merge_atoms(parts)
    if pid in ("3j9i", "6utj", "4cr2"):                             # lay the proteasome barrels along x (side view down z)
        atoms["xyz"] = (atoms["xyz"] @ _ry(np.pi / 2).T).astype(np.float32)
    atoms["xyz"] -= atoms["xyz"].mean(0)
    return atoms


def pdb_cache_dir():
    """Folder (outside the repository) where downloaded PDB files are kept."""
    return os.environ.get("BIO2RESEARCH_PDB_CACHE") or os.path.join(os.path.expanduser("~"), ".cache",
                                                                    "biology_to_research", "pdb")


def _pdb_urls(pid, kind):
    """Download addresses of one PDB file: kind = 'entry' or 'assembly1' (mmCIF) or 'pdb' (legacy PDB format)."""
    pid = pid.lower()
    mid = pid[1:3]
    if kind == "entry":
        return [f"https://files.rcsb.org/download/{pid.upper()}.cif.gz",
                f"https://files.wwpdb.org/pub/pdb/data/structures/divided/mmCIF/{mid}/{pid}.cif.gz",
                f"https://www.ebi.ac.uk/pdbe/entry-files/download/{pid}.cif"]
    if kind == "assembly1":
        return [f"https://files.rcsb.org/download/{pid.upper()}-assembly1.cif.gz",
                f"https://files.wwpdb.org/pub/pdb/data/assemblies/mmCIF/divided/{mid}/{pid}-assembly1.cif.gz"]
    if kind == "pdb":
        return [f"https://files.rcsb.org/download/{pid.upper()}.pdb.gz",
                f"https://files.wwpdb.org/pub/pdb/data/structures/divided/pdb/{mid}/pdb{pid}.ent.gz",
                f"https://www.ebi.ac.uk/pdbe/entry-files/download/pdb{pid}.ent"]
    raise ValueError(kind)


def fetch_pdb(pid, kind="entry", timeout=60):
    """Text of a PDB file, downloaded once and cached outside the repository. Returns (text, where_from), or
    (None, reason) when no mirror can be reached (or BIO2RESEARCH_STANDINS is set)."""
    import gzip
    import urllib.request
    name = {"entry": f"{pid.lower()}.cif.gz", "assembly1": f"{pid.lower()}-assembly1.cif.gz",
            "pdb": f"{pid.lower()}.pdb.gz"}[kind]
    path = os.path.join(pdb_cache_dir(), name)
    if os.path.exists(path):
        return gzip.decompress(open(path, "rb").read()).decode("utf-8", "replace"), f"cached copy ({path})"
    if os.environ.get("BIO2RESEARCH_STANDINS"):
        return None, "BIO2RESEARCH_STANDINS is set"
    reason = "no mirror tried"
    for url in _pdb_urls(pid, kind):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as response:
                raw = response.read()
            text = (gzip.decompress(raw) if raw[:2] == b"\x1f\x8b" else raw).decode("utf-8", "replace")
            if ("_atom_site." not in text) and ("\nATOM  " not in text):
                raise ValueError("no atoms in the downloaded file")
            os.makedirs(pdb_cache_dir(), exist_ok=True)
            with open(path, "wb") as f:
                f.write(gzip.compress(text.encode()))
            return text, url
        except Exception as e:  # noqa: BLE001 — any failure: try the next mirror
            reason = f"{type(e).__name__}: {str(e)[:120]}"
    return None, reason


def _cif_tokens(line):
    """Split one line of an mmCIF loop into values (quoted values may contain spaces)."""
    if "'" not in line and '"' not in line:
        return line.split()
    import re
    toks = re.findall(r"'(?:[^']|'(?=\S))*'|\"(?:[^\"]|\"(?=\S))*\"|\S+", line)
    return [t[1:-1] if len(t) > 1 and t[0] == t[-1] and t[0] in "'\"" else t for t in toks]


_WATER = {"HOH", "DOD", "WAT", "H2O"}
def read_mmcif_atoms(text):
    """Atoms of the first model in an mmCIF file → dict(xyz, name, element, resname, chain, resnum).

    Reads the `_atom_site` loop; skips water and hydrogen, and keeps only the first of alternative atom locations."""
    lines = text.splitlines()
    i = next(k for k, line in enumerate(lines) if line.startswith("_atom_site."))
    cols = []
    while i < len(lines) and lines[i].startswith("_atom_site."):
        cols.append(lines[i].split()[0][len("_atom_site."):])
        i += 1
    c = {name: k for k, name in enumerate(cols)}
    get = lambda *names: next((c[n] for n in names if n in c), None)   # noqa: E731
    ix, iy, iz = c["Cartn_x"], c["Cartn_y"], c["Cartn_z"]
    i_el, i_name = get("type_symbol"), get("auth_atom_id", "label_atom_id")
    i_res, i_ch = get("auth_comp_id", "label_comp_id"), get("auth_asym_id", "label_asym_id")
    i_num, i_alt, i_mod = get("auth_seq_id", "label_seq_id"), get("label_alt_id"), get("pdbx_PDB_model_num")
    xyz, name, element, resname, chain, resnum = [], [], [], [], [], []
    first_model, buf = None, []
    for line in lines[i:]:
        if not line or line.startswith(("#", "_", "loop_", "data_")):
            break
        buf += _cif_tokens(line)
        while len(buf) >= len(cols):
            row, buf = buf[:len(cols)], buf[len(cols):]
            if i_mod is not None:
                first_model = row[i_mod] if first_model is None else first_model
                if row[i_mod] != first_model:
                    continue
            el = row[i_el].upper() if i_el is not None else row[i_name][:1].upper()
            if el in ("H", "D") or row[i_res] in _WATER:
                continue
            if i_alt is not None and row[i_alt] not in (".", "?", "A", "1"):
                continue
            xyz.append((float(row[ix]), float(row[iy]), float(row[iz])))
            name.append(row[i_name])
            element.append(el.capitalize() if len(el) > 1 else el)
            resname.append(row[i_res])
            chain.append(row[i_ch])
            num = row[i_num] if i_num is not None else "0"
            resnum.append(int(num) if num.lstrip("-").isdigit() else 0)
    return dict(xyz=np.array(xyz, np.float32), name=np.array(name), element=np.array(element),
                resname=np.array(resname), chain=np.array(chain), resnum=np.array(resnum))


_ATOMIC_MASS = {"H": 1.008, "C": 12.011, "N": 14.007, "O": 15.999, "P": 30.974, "S": 32.06, "Se": 78.97, "Mg": 24.305,
                "Zn": 65.38, "Fe": 55.845, "Mn": 54.938, "K": 39.098, "Na": 22.99, "Cl": 35.45, "Ca": 40.078}
def model_mass_kDa(atoms):
    """Mass of an atomic model without hydrogens, plus ~8 % for the missing hydrogens (kDa)."""
    return 1.08 * sum(_ATOMIC_MASS.get(e, 12.0) for e in atoms["element"]) / 1000.0


def canonical_frame(atoms):
    """Centre a model and rotate it onto its principal axes — longest extent along x, shortest along z — with the
    signs fixed by the skewness of the atom distribution. A proteasome barrel then lies along x and a ring
    (RuBisCO, p97) lies in the x–y plane, like the stand-ins."""
    xyz = atoms["xyz"].astype(np.float64)
    X = xyz - xyz.mean(0)
    evecs = np.linalg.eigh(np.cov(X.T))[1][:, ::-1]                 # columns: principal axes, largest first
    Y = X @ evecs
    sign = np.sign(np.sum(Y ** 3, axis=0))
    sign[sign == 0] = 1.0
    Y *= sign
    if np.linalg.det(evecs * sign) < 0:                             # keep a proper rotation (no mirror image)
        Y[:, 2] *= -1
    out = dict(atoms)
    out["xyz"] = Y.astype(np.float32)
    return out


def pdb_atoms(pid, verbose=False):
    """Real atomic model of a PDB entry, in the canonical frame. Returns (atoms, description), or (None, reason).

    Uses the deposited entry unless its mass is far from ZOO's reference mass, in which case the first biological
    assembly is tried and the closer of the two kept (e.g. one ribosome rather than the two of a crystal)."""
    pid = pid.lower()
    ref = ZOO.get(pid, {}).get("mass_kDa") or _EXTRA_ENTRIES.get(pid, {}).get("mass_kDa")
    candidates, reason = [], ""
    for kind in ("entry", "assembly1"):
        if candidates and (ref is None or abs(np.log(model_mass_kDa(candidates[0][0]) / ref)) < np.log(1.3)):
            break
        text, where = fetch_pdb(pid, kind)
        if text is None:
            reason = where
            continue
        try:
            atoms = read_mmcif_atoms(text)
        except (StopIteration, KeyError, ValueError) as e:
            reason = f"could not read the file ({type(e).__name__})"
            continue
        if len(atoms["xyz"]):
            label = "deposited entry" if kind == "entry" else "biological assembly 1"
            candidates.append((atoms, f"PDB {pid.upper()} ({label}; {len(atoms['xyz']):,} atoms; from {where})"))
    if not candidates:
        return None, reason
    atoms, desc = min(candidates, key=lambda a: abs(np.log(model_mass_kDa(a[0]) / ref)) if ref else 0.0)
    if verbose:
        print(desc)
    return canonical_frame(atoms), desc


_ATOMS_CACHE = {}
def zoo_atoms(pid):
    """Atomic model (dict of arrays, Å, centred) of zoo entry `pid`: the real PDB model when it can be downloaded, the
    code-built stand-in otherwise. The source is recorded in `atoms_source(pid)`."""
    pid = pid.lower()
    if pid not in _ATOMS_CACHE:
        atoms, desc = pdb_atoms(pid)
        if atoms is None:
            atoms, desc = standin_atoms(pid), f"code-built stand-in modelled on PDB {pid.upper()} (download failed: {desc})"
        _ATOMS_CACHE[pid] = (atoms, desc)
    return _ATOMS_CACHE[pid][0]


def atoms_source(pid):
    """Where the atoms of zoo entry `pid` came from (a real PDB file or the offline stand-in)."""
    zoo_atoms(pid)
    return _ATOMS_CACHE[pid.lower()][1]


_Z = {"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0, "P": 15.0, "H": 1.0, "Mg": 12.0, "Zn": 30.0, "Fe": 26.0, "K": 19.0,
      "Na": 11.0, "Cl": 17.0, "Ca": 20.0, "Mn": 25.0, "Se": 34.0}
def atoms_to_map(xyz, weights=None, voxel=10.0, box=None, sigma_A=None):
    """Density map [z, y, x] from atom positions (Å): trilinear splatting on a grid twice as fine, Gaussian blur
    (σ = `sigma_A`, default = voxel/2), then 2×2×2 binning to `voxel` Å. Centred on the box if `box` is given."""
    xyz = np.asarray(xyz, float)
    w = np.ones(len(xyz)) if weights is None else np.asarray(weights, float)
    fine = voxel / 2.0
    if box is None:
        ext = np.ptp(xyz, 0).max() + 6 * voxel
        box = int(np.ceil(ext / voxel)) + 1
    n = 2 * box
    centre = (n - 1) / 2.0
    p = xyz[:, ::-1] / fine + centre                                  # (z, y, x) grid coordinates
    grid = np.zeros((n, n, n), np.float32)
    i0 = np.floor(p).astype(int)
    f = p - i0
    for dz in (0, 1):
        for dy in (0, 1):
            for dx in (0, 1):
                idx = i0 + np.array([dz, dy, dx])
                wt = w * np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1]) * np.where(dx, f[:, 2], 1 - f[:, 2])
                ok = np.all((idx >= 0) & (idx < n), 1)
                np.add.at(grid, tuple(idx[ok].T), wt[ok])
    grid = ndimage.gaussian_filter(grid, (sigma_A if sigma_A else voxel / 2.0) / fine)
    return grid.reshape(box, 2, box, 2, box, 2).mean((1, 3, 5)).astype(np.float32)


def zoo_map(pid, voxel="10A"):
    """(density[z, y, x], voxel size Å, source) of a zoo entry, painted from its atoms (`zoo_atoms`)."""
    atoms = zoo_atoms(pid)
    v = float(str(voxel).rstrip("A"))
    w = np.array([_Z.get(e, 6.0) for e in atoms["element"]])
    vol = atoms_to_map(atoms["xyz"], w, voxel=v)
    return vol, v, atoms_source(pid)


def betagal_map(voxel=3.54, box=100, b_factor=40.0):
    """E. coli β-galactosidase (PDB 5A1A, a D2 homotetramer, ~4 × 1,020 residues) at `voxel` Å in a `box`³ grid.

    Returns (map[z, y, x], voxel, source, atoms). Atomic B-factor `b_factor` (Å²) blurs each atom by
    σ = √(B / 8π²); helices (≈ 10 Å apart) are visible at this sampling, side chains are not. Offline, a code-built
    β-galactosidase-sized D2 tetramer is used instead.
    """
    atoms, source = pdb_atoms("5a1a")
    if atoms is None:
        atoms, source = _betagal_standin_atoms(), f"code-built β-galactosidase-sized D2 tetramer (download failed: {source})"
    w = np.array([_Z.get(e, 6.0) for e in atoms["element"]])
    sigma = np.sqrt(b_factor / (8 * np.pi ** 2) + (voxel / 2.0) ** 2)                  # B-factor + sampling blur
    vol = atoms_to_map(atoms["xyz"], w, voxel=voxel, box=box, sigma_A=sigma)
    return vol, voxel, source, atoms


def _betagal_standin_atoms():
    """Code-built β-galactosidase-sized D2 homotetramer (chains A–D), centred at the origin."""
    rng = np.random.default_rng(1234)
    domains = [((21.0, 18.0, 22.0), (0.0, 0.0, 0.0)), ((18.0, 16.0, 20.0), (34.0, 8.0, 4.0)),
               ((17.0, 15.0, 19.0), (-31.0, 10.0, -3.0)), ((16.0, 15.0, 18.0), (6.0, 31.0, 8.0)),
               ((15.0, 14.0, 17.0), (-8.0, -29.0, 5.0))]
    parts = [_transform(helix_bundle(r, rng), _rz(rng.uniform(0, 6.3)), c) for r, c in domains]
    mono = merge_atoms(parts)
    mono["xyz"] = mono["xyz"] - mono["xyz"].mean(0) + np.array([30.0, 21.0, 13.0])
    subunits = []
    for k, R in enumerate([np.eye(3), _rz(np.pi), _rx(np.pi), _ry(np.pi)]):          # D2: identity + three 2-folds
        s = _transform(mono, R)
        s["chain"] = np.full(len(s["xyz"]), "ABCD"[k])
        subunits.append(s)
    atoms = merge_atoms(subunits)
    atoms["xyz"] -= atoms["xyz"].mean(0)
    return atoms


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    Zoo maps are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)                  # where the crop starts in the input ...
    dst0 = np.maximum((size - s) // 2, 0)                  # ... and where the pad starts in the output
    n = np.minimum(s, size)
    com = np.array(ndimage.center_of_mass(v))
    shift = (size - 1) / 2 - dst0 + src0 - com             # puts the centre of mass exactly at the output's centre
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def load_zoo(ids=("4v4r", "3j9i", "6utj", "1bxn"), size: int = 32, sigma: float = 1.0,
             verbose: bool = True) -> dict:
    """Build and prepare several zoo maps. Returns {pdb_id: volume(size³)}."""
    zoo = {}
    for pid in ids:
        raw, vs, src = zoo_map(pid)
        zoo[pid] = prepare_map(raw, size=size, sigma=sigma)
        if verbose:
            print(f"  {pid.upper():5s} {ZOO.get(pid, {}).get('name', ''):28s} raw {raw.shape} @ {vs:.0f} Å/voxel  ← {src}")
    return zoo


def rotate_image(img: np.ndarray, angle_rad: float, shift_xy=(0, 0), order: int = 1) -> np.ndarray:
    """2D analogue of :func:`rotate_volume` (counter-clockwise in the (x, y) plane)."""
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    R = np.array([[c, -s], [s, c]])
    R_idx = R[::-1, ::-1]                      # (x,y) -> (y,x)
    ctr = (np.array(img.shape) - 1) / 2.0
    t_idx = np.asarray(shift_xy, float)[::-1]
    M = R_idx.T
    return ndimage.affine_transform(img, M, offset=ctr - M @ (ctr + t_idx), order=order, mode="constant")


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def radial_frequency(shape) -> np.ndarray:
    """|k| in cycles/voxel for every coefficient of an unshifted FFT."""
    return np.sqrt(sum(k**2 for k in freq_grid(shape)))


def missing_wedge_mask(shape, tilt_max_deg: float = 60.0) -> np.ndarray:
    """Binary Fourier mask of a single-axis tilt series about y (±tilt_max).

    A coefficient (k_z, k_y, k_x) is measured if |k_z| ≤ |k_x|·tan(tilt_max)
    — the formula M(ξ) = 1(|ξ_3| ≤ |ξ_1| tan θ) used by Xu et al. (2012).
    For 2D images [z, x] pass a 2-tuple shape.
    """
    k = freq_grid(shape)
    kz, kx = k[0], k[-1]
    m = np.abs(kz) <= np.abs(kx) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[(0,) * len(shape)] = True               # keep the mean (DC) term
    return m.astype(np.float32)


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def lowpass(vol: np.ndarray, cutoff: float, soft: float = 0.02) -> np.ndarray:
    """Soft low-pass filter; ``cutoff`` in cycles/voxel (0.5 = Nyquist)."""
    k = radial_frequency(vol.shape)
    filt = 1 / (1 + np.exp((k - cutoff) / soft))
    return apply_fourier_mask(vol, filt)


def add_noise_snr(signal: np.ndarray, snr: float, rng: np.random.Generator) -> np.ndarray:
    """Add white Gaussian noise so that var(signal)/var(noise) = snr."""
    sigma = np.sqrt(signal.var() / snr)
    return (signal + rng.normal(0, sigma, signal.shape)).astype(np.float32)


def pearson(a: np.ndarray, b: np.ndarray) -> float:
    """Pearson correlation of two arrays (flattened)."""
    a = a.ravel() - a.mean()
    b = b.ravel() - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def fsc(v1: np.ndarray, v2: np.ndarray, pixel_A: float = 10.0, n_shells: int | None = None):
    """Fourier shell (ring, in 2D) correlation. Returns (frequency_1/Å, fsc, n_coeffs)."""
    F1, F2 = np.fft.fftn(v1), np.fft.fftn(v2)
    k = radial_frequency(v1.shape)
    n_shells = n_shells or min(v1.shape) // 2
    idx = np.minimum((k * 2 * n_shells).astype(int), n_shells)
    num = np.bincount(idx.ravel(), (F1 * np.conj(F2)).real.ravel(), n_shells + 1)
    d1 = np.bincount(idx.ravel(), (np.abs(F1)**2).ravel(), n_shells + 1)
    d2 = np.bincount(idx.ravel(), (np.abs(F2)**2).ravel(), n_shells + 1)
    cnt = np.bincount(idx.ravel(), minlength=n_shells + 1)
    curve = num / np.sqrt(d1 * d2 + 1e-20)
    freq = (np.arange(n_shells + 1) + 0.5) / (2 * n_shells) / pixel_A
    return freq[:n_shells], curve[:n_shells], cnt[:n_shells]


def resolution_at(freq, curve, threshold: float = 0.143) -> float:
    """First crossing of the FSC curve below ``threshold`` → resolution in Å."""
    below = np.where(curve[1:] < threshold)[0]
    if len(below) == 0:
        return float(1 / freq[-1])
    i = below[0] + 1
    f0, f1, c0, c1 = freq[i - 1], freq[i], curve[i - 1], curve[i]
    f = f0 + (threshold - c0) * (f1 - f0) / (c1 - c0 + 1e-12)
    return float(1 / f)
print("🧰 toolkit loaded")

In [ ]:
# ── Data: zoo maps (sharper, 0.5-voxel blur) and a high-resolution map of β-galactosidase ──
zoo = load_zoo(["4v4r", "3j9i", "1bxn"], size=32, sigma=0.5)


def projection_64(pid: str) -> np.ndarray:
    """Projection of a zoo map along z, placed in the centre of a 64 × 64 image (1 nm pixels), scaled to max 1."""
    img = np.zeros((64, 64), np.float32)
    img[16:48, 16:48] = zoo[pid].sum(axis=0)
    return img / img.max()


ribo2d, prot2d, rubisco2d = projection_64("4v4r"), projection_64("3j9i"), projection_64("1bxn")
bgal_map, bgal_voxel, bgal_src, _ = betagal_map()            # Module 01, Section 4
print(f"β-galactosidase: {bgal_map.shape} @ {bgal_voxel} Å/voxel  ← {bgal_src}")

---
## 1 · Waves: amplitude, frequency and phase

> 📍 **Where we are.** Modules 01–02 treated maps as grids of numbers and taught you to rotate them. This module
> introduces a second, equally important way to look at the *same* numbers: as a sum of **waves**. Modules 06–10
> (imaging, tomography, resolution) and the fast algorithms of Modules 11–13 and 21 are built on it.

A **wave** (here a cosine) along a line is described by three numbers:

$$f(x) = A\cos\big(2\pi k x + \varphi\big)$$

- **amplitude** $A$: how strong it is (height of the crests);
- **spatial frequency** $k$: how many cycles per unit length (cycles/Å); its inverse is the **wavelength**
  $\lambda = 1/k$ (Å per cycle);
- **phase** $\varphi$ (radians): where the crests are. Shifting the wave by a distance $d$ changes the phase by
  $-2\pi k d$.

🧫 Biology is full of near-periodic structure: the α-helix rises 5.4 Å per turn (its *pitch*), neighbouring strands in a
β-sheet are ≈ 4.8 Å apart, packed helices ≈ 10 Å apart, and the two head-group layers of a lipid bilayer ≈ 40 Å apart.
Each of these is a "frequency" that a map either contains or not.

In [ ]:
# ── 1.1 Amplitude, frequency and phase ──
x = np.linspace(0, 40, 1000)                                  # position along a line (Å)
fig, axes = plt.subplots(1, 3, figsize=(11, 2.8), sharey=True)
for A in [0.5, 1.0]:
    axes[0].plot(x, A * np.cos(2 * np.pi * x / 10), label=f"A = {A}")
for lam in [20, 10, 5.4]:
    axes[1].plot(x, np.cos(2 * np.pi * x / lam), label=f"λ = {lam} Å (k = {1 / lam:.3f} Å⁻¹)")
for phi in [0, np.pi / 2, np.pi]:
    axes[2].plot(x, np.cos(2 * np.pi * x / 10 + phi), label=f"φ = {phi / np.pi:.1f}π")
for ax, t in zip(axes, ["amplitude: how strong", "frequency: how fine", "phase: where"]):
    ax.set_title(t); ax.set_xlabel("position x (Å)"); ax.legend(fontsize=7, loc="lower right")
axes[0].set_ylabel("f(x)")
plt.tight_layout(); plt.show()

---
## 2 · The Fourier transform in 1D: how much of each wave is in a signal?

🧮 **Complex numbers as arrows.** The Fourier transform needs both an amplitude and a phase for every frequency. A
**complex number** $z = a + ib$ (with $i^2 = -1$) packs both: think of it as an arrow in a plane, with **length**
$|z| = \sqrt{a^2 + b^2}$ (the amplitude) and **angle** $\arg z$ (the phase). Euler's formula
$e^{i\theta} = \cos\theta + i\sin\theta$ is an arrow of length 1 at angle θ; multiplying by it *rotates* an arrow by θ.

The **discrete Fourier transform (DFT)** of $N$ samples $f_0, \dots, f_{N-1}$ is

$$F_k = \sum_{n=0}^{N-1} f_n\, e^{-2\pi i\,kn/N},\qquad k = 0, 1, \dots, N-1.$$

*In words:* for each frequency $k$ (in cycles per $N$ samples), multiply the signal by a wave of that frequency and add
up — a measure of "how much of this wave, at which phase, the signal contains". A cosine $A\cos(2\pi k_0 n/N + \varphi)$
shows up as $|F_{k_0}| = NA/2$ with $\arg F_{k_0} = \varphi$. The transform is perfectly reversible (inverse DFT), and
the **FFT** (fast Fourier transform) computes it in about $N\log N$ steps instead of $N^2$.

Practical NumPy facts: `np.fft.fft(f)` returns $F_k$; `np.fft.fftfreq(N, d)` gives the frequency of each $k$ in cycles
per unit of `d` (the sample spacing) — the second half are **negative frequencies**; for real signals
$F_{-k} = \overline{F_k}$, so the negative half mirrors the positive half. `np.fft.fftshift` moves frequency 0 to the
middle for plotting.

In [ ]:
# ── 2.1 Find the hidden waves in a noisy profile ──
dx = 1.0                                               # sample spacing: 1 Å
n_samples = 200
xs = np.arange(n_samples) * dx
signal = 1.0 * np.cos(2 * np.pi * xs / 40) + 0.5 * np.cos(2 * np.pi * xs / 10 + 1.0)   # 40 Å and 10 Å waves
noisy_profile = signal + rng.normal(0, 0.7, n_samples)

F = np.fft.fft(noisy_profile)
freqs = np.fft.fftfreq(n_samples, d=dx)                # cycles per Å
amplitude = 2 * np.abs(F) / n_samples                  # convert |F_k| back to the wave's amplitude A
for lam in [40, 10]:
    k_idx = int(round(n_samples * dx / lam))
    print(f"wave of λ = {lam:2d} Å: k = {freqs[k_idx]:.3f} Å⁻¹, recovered amplitude {amplitude[k_idx]:.2f}, "
          f"phase {np.angle(F[k_idx]):+.2f} rad")
print("inverse FFT gives back the signal:", np.allclose(np.fft.ifft(F).real, noisy_profile))
print(f"energy is conserved (Parseval): Σ|f|² = {np.sum(noisy_profile**2):.1f}, "
      f"Σ|F|²/N = {np.sum(np.abs(F)**2) / n_samples:.1f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].plot(xs, noisy_profile, color="0.6", lw=1, label="noisy measurement")
axes[0].plot(xs, signal, color="tab:blue", label="hidden signal (40 Å + 10 Å waves)")
axes[0].set_xlabel("position (Å)"); axes[0].set_ylabel("density (a.u.)"); axes[0].legend(fontsize=7)
axes[0].set_title("real space: hard to read")
axes[1].stem(np.fft.fftshift(freqs), np.fft.fftshift(amplitude), basefmt=" ", markerfmt=".")
axes[1].set_xlabel("spatial frequency k (Å⁻¹; negative half mirrors positive half)")
axes[1].set_ylabel("amplitude 2|F_k|/N"); axes[1].set_title("Fourier space: two clear peaks at ±1/40 and ±1/10 Å⁻¹")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Two waves buried in noise became two sharp peaks: noise spreads its energy thinly over *all*
frequencies (it is "white"), while a wave concentrates its energy at one frequency. The FFT also recovered each wave's
amplitude (≈ 1.0 and ≈ 0.5) and the 10 Å wave's phase (≈ 1 rad). This is the deep reason why averaging and filtering
work better in Fourier space.

---
## 3 · Building a profile from waves: how many waves resolve a membrane?

Any profile can be written as a sum of waves (a **Fourier series**). Keeping only waves up to a maximum frequency
$k_{\max}$ gives a smoothed version — exactly what a map "at resolution $d = 1/k_{\max}$" means. Take the density
across a lipid bilayer: two dense head-group layers ≈ 40 Å apart with the less dense lipid tails between them. How fine
must the waves be before the two layers appear as two separate peaks?

In [ ]:
# ── 3.1 A membrane profile rebuilt from its lowest frequencies ──
xm = np.arange(-128, 128) * 1.0                                         # 256 samples, 1 Å apart
membrane = np.exp(-(xm - 20)**2 / (2 * 4**2)) + np.exp(-(xm + 20)**2 / (2 * 4**2))   # two head-group layers
Fm = np.fft.fft(membrane)
km = np.fft.fftfreq(xm.size, d=1.0)


def keep_below(F_: np.ndarray, k_: np.ndarray, k_max: float) -> np.ndarray:
    """Inverse FFT keeping only frequencies |k| ≤ k_max (a sharp low-pass)."""
    return np.fft.ifft(F_ * (np.abs(k_) <= k_max)).real


fig, ax = plt.subplots(figsize=(7.5, 3.2))
ax.plot(xm, membrane, color="k", lw=2, label="true profile")
for res in [120, 60, 40, 25, 10]:
    ax.plot(xm, keep_below(Fm, km, 1 / res), lw=1.2, label=f"waves down to λ = {res} Å")
ax.set_xlim(-80, 80); ax.set_xlabel("distance across the membrane (Å)"); ax.set_ylabel("density (a.u.)")
ax.set_title("A bilayer at different resolutions"); ax.legend(fontsize=7, ncol=2); plt.show()

🔍 **What just happened?** With only long waves (≥ 60 Å) the bilayer is one blob; once waves of about the head-group
spacing (≈ 40 Å) are included, two layers appear, and finer waves sharpen them. **Resolution is the shortest wavelength
you can trust.** Notice also the small wiggles near the peaks with a sharp cut-off: that is **ringing** (Section 6).

✏️ **Your turn.** Move the head groups closer (replace `20` by `12` in both places — a thinner membrane) and find the
coarsest resolution at which you still see two peaks. What simple rule relates the spacing to the resolution needed?

---
## 4 · 2D Fourier transforms: images as sums of plane waves

In 2D, a wave has a frequency **and a direction**: $\cos\big(2\pi(k_x x + k_y y) + \varphi\big)$ looks like a stripe
pattern (a *grating*) whose stripes run perpendicular to the vector $(k_x, k_y)$, with spacing $1/\sqrt{k_x^2 + k_y^2}$.
The 2D FFT (`np.fft.fft2`; `fftn` for any dimension) gives one complex number per $(k_y, k_x)$. After `fftshift`, the
centre of the spectrum is frequency 0 (the image mean), and distance from the centre = frequency. A single grating
therefore appears as **two dots** (at $\pm(k_x, k_y)$).

In [ ]:
# ── 4.1 Gratings and their spectra ──
yy, xx = np.mgrid[0:64, 0:64]
gratings = {"vertical stripes, 8 px": np.cos(2 * np.pi * xx / 8),
            "horizontal stripes, 16 px": np.cos(2 * np.pi * yy / 16),
            "diagonal, fine": np.cos(2 * np.pi * (xx / 6 + yy / 6)),
            "sum of the three": np.cos(2 * np.pi * xx / 8) + np.cos(2 * np.pi * yy / 16) + np.cos(2 * np.pi * (xx + yy) / 6)}
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for j, (name, g) in enumerate(gratings.items()):
    axes[0, j].imshow(g); axes[0, j].set_title(name, fontsize=8); axes[0, j].axis("off")
    spec = np.abs(np.fft.fftshift(np.fft.fft2(g)))
    axes[1, j].imshow(spec, cmap="viridis", extent=(-0.5, 0.5, 0.5, -0.5))
    axes[1, j].set_xlabel("k_x (cycles/px)", fontsize=8)
axes[1, 0].set_ylabel("k_y (cycles/px)", fontsize=8)
fig.suptitle("Top: images. Bottom: |FFT| after fftshift (centre = frequency 0)", fontsize=10)
plt.tight_layout(); plt.show()

(The faint crosses around the diagonal grating's dots are **spectral leakage**: 64 pixels do not hold a whole number of
6-pixel periods, so the FFT sees a small jump at the image edge.)

Real images contain all frequencies at once. The **power spectrum** $|F|^2$ (shown on a log scale because it spans many
orders of magnitude) reveals how the image's energy is distributed: smooth shapes put most energy near the centre (low
frequencies), sharp edges and fine details need high frequencies, and white noise adds a flat floor everywhere.

In [ ]:
# ── 4.2 Power spectra of molecules, with and without noise ──
noisy_ribo2d = add_noise_snr(ribo2d, 1.0, rng)
panels = [(ribo2d, "ribosome projection"), (prot2d, "proteasome projection"), (noisy_ribo2d, "ribosome + noise (SNR 1)")]
fig, axes = plt.subplots(2, 3, figsize=(8.5, 5.4))
for j, (im, name) in enumerate(panels):
    axes[0, j].imshow(im); axes[0, j].set_title(name, fontsize=9); axes[0, j].axis("off")
    ps = np.log10(np.abs(np.fft.fftshift(np.fft.fft2(im)))**2 + 1e-6)
    axes[1, j].imshow(ps, cmap="viridis", vmin=-3, vmax=ps.max(), extent=(-0.5, 0.5, 0.5, -0.5))
    axes[1, j].set_title("log₁₀ power spectrum", fontsize=9); axes[1, j].set_xlabel("k_x (cycles/nm)", fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The molecules' power is concentrated near the centre (they are smooth at 1 nm pixels) and
decays outwards; the elongated proteasome has a spectrum stretched in the direction *perpendicular* to its long axis
(narrow in real space ↔ wide in Fourier space). Noise lifts the whole spectrum to a flat floor: beyond the frequency
where the molecule's power sinks below the floor, the image contains no usable information — that is, in essence,
what *resolution* measures (Section 11 and Module 10).

---
## 5 · Frequency ↔ resolution in ångströms

Frequencies from `fftfreq` are in **cycles per pixel**. Dividing by the pixel size converts them to cycles per Å, and
the corresponding **resolution** is the wavelength:

$$k\ [\text{Å}^{-1}] = \frac{k\ [\text{cycles/pixel}]}{\text{pixel size}\ [\text{Å}]},\qquad d\ [\text{Å}] = \frac1k.$$

The highest frequency a grid can hold is 0.5 cycles/pixel (one crest and one trough every two pixels): the **Nyquist
frequency**. So the best possible resolution is $d_{\min} = 2\times$ pixel size (20 Å for our 10-Å zoo maps, 7.08 Å
for β-galactosidase at 3.54 Å/voxel). Waves finer than Nyquist do not disappear when sampled — they masquerade as
*coarser* waves (**aliasing**), which is why images are low-pass filtered before binning.

In [ ]:
# ── 5.1 Aliasing: a fine wave sampled too coarsely ──
x_fine = np.linspace(0, 40, 2000)
true_wave = np.cos(2 * np.pi * x_fine / 3.0)                       # a 3 Å periodicity
pixel = 2.5                                                         # sampled every 2.5 Å (Nyquist limit 5 Å)
x_samp = np.arange(0, 40.01, pixel)
alias_k = abs(1 / 3.0 - round(pixel / 3.0) / pixel)                 # folded frequency (cycles/Å)
fig, ax = plt.subplots(figsize=(7.5, 2.8))
ax.plot(x_fine, true_wave, color="0.7", label="true 3 Å wave")
ax.plot(x_samp, np.cos(2 * np.pi * x_samp / 3.0), "o", color="tab:red", label="samples every 2.5 Å")
ax.plot(x_fine, np.cos(2 * np.pi * alias_k * x_fine), color="tab:red", ls="--",
        label=f"what the samples show: a {1 / alias_k:.0f} Å wave")
ax.set_xlabel("position (Å)"); ax.set_title("Aliasing: finer than Nyquist ⇒ a fake coarse wave")
ax.legend(fontsize=7, loc="lower right"); plt.show()

In [ ]:
# ── 5.2 A high-resolution map at 30, 15 and 8 Å ──
print("frequency index → resolution for a 100-voxel box at 3.54 Å/voxel:")
for j in [1, 5, 10, 25, 50]:
    k_cpp = j / bgal_map.shape[0]
    print(f"  index {j:3d}: {k_cpp:.2f} cycles/voxel = {k_cpp / bgal_voxel:.4f} Å⁻¹ → d = {bgal_voxel / k_cpp:6.1f} Å")
fig, axes = plt.subplots(1, 4, figsize=(10.5, 2.9))
mid = bgal_map.shape[0] // 2
for ax, res in zip(axes, [None, 8, 15, 30]):
    v = bgal_map if res is None else lowpass(bgal_map, cutoff=bgal_voxel / res, soft=0.01)
    ax.imshow(v[mid, 20:80, 20:80]); ax.axis("off")
    ax.set_title("original (Nyquist 7.1 Å)" if res is None else f"low-passed to {res} Å", fontsize=9)
fig.suptitle("β-galactosidase central slice: detail disappears as high frequencies are removed", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Removing frequencies above $1/d$ is the *definition* of showing a map "at $d$ Å". At 8 Å this
slice barely changes (the test map holds little detail finer than that); at 15 Å the internal texture fades; at 30 Å only
the outline and the central hole remain. Rough rules of thumb used in the
field: ≈ 20–30 Å shows overall shape and domains; ≈ 8–10 Å shows α-helices as rods; ≈ 4–5 Å separates β-strands;
≈ 3–3.5 Å shows many side chains. Cellular tomograms are typically interpreted at 20–50 Å; subtomogram averages reach
from tens of Å down to a few Å.

---
## 6 · Filters: keeping some frequencies, discarding others

A **Fourier filter** multiplies the transform by a mask $H(k)$ and transforms back:
$f_{\text{filtered}} = \mathcal F^{-1}[H \cdot \mathcal F f]$ (the toolkit's `apply_fourier_mask`). Three classics:

| filter | keeps | effect | use |
|---|---|---|---|
| **low-pass** | $\lvert k\rvert <$ cutoff | smooths, removes noise and fine detail | display, denoising, "filter to 40 Å" |
| **high-pass** | $\lvert k\rvert >$ cutoff | removes slow background gradients, keeps edges | flatten uneven ice thickness |
| **band-pass** | a ring of frequencies | keeps features of one size | particle picking (Module 11) |

⚠️ A **hard** (step) cutoff creates **ringing** — ripples next to sharp edges (the Gibbs phenomenon), because a sharp
edge in Fourier space is a wiggly function in real space. **Soft** edges (e.g. a Gaussian or the toolkit's
sigmoid-shaped `lowpass`) avoid it.

In [ ]:
# ── 6.1 Low-, high-, band-pass and a hard low-pass on a noisy ribosome image ──
k2d = radial_frequency(noisy_ribo2d.shape)                       # |k| in cycles/pixel (unshifted layout)
masks = {"soft low-pass (k < 0.10)": 1 / (1 + np.exp((k2d - 0.10) / 0.015)),
         "soft high-pass (k > 0.10)": 1 - 1 / (1 + np.exp((k2d - 0.10) / 0.015)),
         "band-pass (0.03–0.10)": np.exp(-k2d**2 / (2 * 0.10**2)) - np.exp(-k2d**2 / (2 * 0.03**2)),
         "HARD low-pass (k < 0.10)": (k2d < 0.10).astype(float)}
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for j, (name, H) in enumerate(masks.items()):
    axes[0, j].imshow(np.fft.fftshift(H), cmap="viridis", extent=(-0.5, 0.5, 0.5, -0.5), vmin=0, vmax=1)
    axes[0, j].set_title(name, fontsize=8); axes[0, j].set_xlabel("k_x (cycles/nm)", fontsize=7)
    axes[1, j].imshow(apply_fourier_mask(noisy_ribo2d, H)[8:56, 8:56]); axes[1, j].axis("off")
plt.tight_layout(); plt.show()

# a close look at ringing: a sharp-edged disc filtered with a hard and a soft cutoff
disc = ((xx - 31.5)**2 + (yy - 31.5)**2 < 12**2).astype(float)
hard = apply_fourier_mask(disc, (k2d < 0.12).astype(float))
soft = apply_fourier_mask(disc, 1 / (1 + np.exp((k2d - 0.12) / 0.02)))
print(f"overshoot above 1 next to the disc edge — hard cutoff: {hard.max() - 1:+.3f}, soft cutoff: {soft.max() - 1:+.3f}")

🔍 **What just happened?** The low-pass removed most of the speckle and kept the molecule; the high-pass kept only edges
and noise; the band-pass kept blobs of roughly particle size. The hard cutoff produces faint ripples around the
molecule; on a sharp-edged disc it overshoots measurably, the soft one much less (numbers above). In cryo-EM software,
filters are almost always soft.

---
## 7 · The convolution theorem: blurring is multiplication

Blurring an image with a kernel $h$ (for example, the microscope's point-spread function) is a **convolution**: each
output pixel is a weighted sum of its neighbours, $(f \ast h)(x) = \sum_y f(y)\,h(x - y)$. The **convolution theorem**
says that in Fourier space this becomes a simple product:

$$\mathcal F[f \ast h] = \mathcal F[f]\cdot\mathcal F[h].$$

*In words:* blurring multiplies every frequency by a factor $H(k)$ — the **transfer function** of the blur. A Gaussian
kernel of width $\sigma$ has a Gaussian transfer function of width $1/(2\pi\sigma)$: wide in real space ↔ narrow in
Fourier space. This is how the microscope's contrast transfer function (CTF, Module 06) and the missing wedge (Module 07)
act on images: as multiplications in Fourier space. (Strictly, the FFT version is a *circular* convolution — the image
wraps around at its edges.)

In [ ]:
# ── 7.1 Verify the convolution theorem with a Gaussian blur ──
sigma_px = 2.0
kernel = np.exp(-((xx - 32)**2 + (yy - 32)**2) / (2 * sigma_px**2))
kernel /= kernel.sum()
H_blur = np.fft.fft2(np.fft.ifftshift(kernel))                 # kernel centre moved to index (0, 0)
blur_fourier = np.real(np.fft.ifft2(np.fft.fft2(ribo2d) * H_blur))
blur_direct = ndimage.gaussian_filter(ribo2d, sigma_px, mode="wrap", truncate=8)
print(f"max difference between real-space and Fourier-space blurring: {np.abs(blur_fourier - blur_direct).max():.2e}")
print(f"transfer function at k = 0.08 cycles/px: measured {np.abs(np.fft.fftshift(H_blur))[32, 32 + 5]:.3f}, "
      f"theory exp(−2π²σ²k²) = {np.exp(-2 * np.pi**2 * sigma_px**2 * (5 / 64)**2):.3f}")

fig, axes = plt.subplots(1, 4, figsize=(10.5, 2.7))
axes[0].imshow(kernel[24:40, 24:40]); axes[0].set_title("kernel h (σ = 2 px)", fontsize=9)
axes[1].imshow(np.abs(np.fft.fftshift(H_blur)), cmap="viridis", extent=(-0.5, 0.5, 0.5, -0.5))
axes[1].set_title("|H(k)|: also a Gaussian", fontsize=9)
axes[2].imshow(blur_direct); axes[2].set_title("blur by convolution", fontsize=9)
axes[3].imshow(blur_fourier); axes[3].set_title("blur by multiplying FFTs", fontsize=9)
for ax in axes[[0, 2, 3]]:
    ax.axis("off")
plt.tight_layout(); plt.show()
assert np.abs(blur_fourier - blur_direct).max() < 1e-6

---
## 8 · Cross-correlation: finding a shift, finding particles

How far must image $g$ be moved to best match image $f$? The **cross-correlation** scores every possible shift $t$:

$$c(t) = \sum_x f(x)\, g(x + t) \quad\Longrightarrow\quad c = \mathcal F^{-1}\big[\overline{\mathcal F f}\cdot \mathcal F g\big].$$

*In words:* slide one image over the other, multiply and add at each offset; by the same algebra as the convolution
theorem, all offsets can be computed at once with three FFTs. The position of the highest peak is the shift. Because
the FFT is circular, peak indices above $N/2$ mean negative shifts. This one formula is the engine of **template
matching** (Module 11), **translational alignment** (Modules 12–13) and **tilt-series alignment** (Module 07).

In [ ]:
# ── 8.1 Recover the shift between two noisy copies, and compare speed ──
true_shift = (7, -5)                                                   # (dy, dx) in pixels
moved = np.roll(ribo2d, true_shift, axis=(0, 1))
f_img, g_img = add_noise_snr(ribo2d, 2.0, rng), add_noise_snr(moved, 2.0, rng)


def fft_xcorr(f: np.ndarray, g: np.ndarray) -> np.ndarray:
    """Circular cross-correlation c(t) = Σ f(x) g(x + t) for all shifts t, via FFTs (zero-mean inputs)."""
    return np.real(np.fft.ifft2(np.conj(np.fft.fft2(f - f.mean())) * np.fft.fft2(g - g.mean())))


def peak_shift(c: np.ndarray) -> tuple[int, int]:
    """Location of the highest peak of a circular correlation map, as a signed (dy, dx) shift."""
    i = np.array(np.unravel_index(np.argmax(c), c.shape))
    n = np.array(c.shape)
    return tuple(int(v) for v in (i + n // 2) % n - n // 2)


t0 = time.perf_counter()
cc = fft_xcorr(f_img, g_img)
t_fft = time.perf_counter() - t0
t0 = time.perf_counter()
cc_direct = np.zeros_like(cc)
fz, gz = f_img - f_img.mean(), g_img - g_img.mean()
for dy in range(64):
    for dx in range(64):
        cc_direct[dy, dx] = np.sum(fz * np.roll(gz, (-dy, -dx), axis=(0, 1)))   # Σ f(x) g(x + t)
t_direct = time.perf_counter() - t0
print(f"true shift {true_shift}, FFT estimate {peak_shift(cc)}, direct estimate {peak_shift(cc_direct)}")
trials = [peak_shift(fft_xcorr(add_noise_snr(ribo2d, 2.0, rng), add_noise_snr(moved, 2.0, rng))) for _ in range(20)]
print(f"over 20 fresh noise realisations: exact in {np.mean([t == true_shift for t in trials]):.0%}, "
      f"within ±1 px in {np.mean([max(abs(a - b) for a, b in zip(t, true_shift)) <= 1 for t in trials]):.0%}")
print(f"same correlation map: {np.allclose(cc, cc_direct, atol=1e-3)}; time FFT {t_fft * 1e3:.2f} ms vs "
      f"direct {t_direct * 1e3:.0f} ms")

fig, axes = plt.subplots(1, 3, figsize=(9, 2.9))
axes[0].imshow(f_img); axes[0].set_title("image f (noisy)", fontsize=9)
axes[1].imshow(g_img); axes[1].set_title(f"image g = f moved by {true_shift} + noise", fontsize=9)
axes[2].imshow(np.fft.fftshift(cc), cmap="viridis", extent=(-32, 32, 32, -32))
axes[2].set_title("cross-correlation c(t) (centre = no shift)", fontsize=9)
axes[2].set_xlabel("shift dx (px)"); axes[2].set_ylabel("shift dy (px)")
for ax in axes[:2]:
    ax.axis("off")
plt.tight_layout(); plt.show()
assert max(abs(a - b) for a, b in zip(peak_shift(cc), true_shift)) <= 1, "shift should be found to ±1 px"

🔍 **What just happened?** The FFT gave exactly the same correlation map as the slow double loop, hundreds of times
faster, and its peak sits at (or within one pixel of) the true shift. Why not always exactly? A smooth molecule has a
*broad* correlation peak, and noise can nudge its top by a pixel — repeating the experiment with fresh noise shows how
often. In practice one filters out noise-dominated high frequencies first, refines the peak to sub-pixel precision, or
averages many measurements (Module 04).

The same computation finds **many** particles at once: correlate a large "micrograph" with a small template of the
molecule. Every place where the template fits gives a peak. (Real template matching also searches over rotations and
normalises the score locally — Module 11.)

In [ ]:
# ── 8.2 A preview of template matching: find four ribosomes in a noisy field ──
field = np.zeros((128, 128), np.float32)
true_pos = [(30, 35), (40, 95), (90, 25), (100, 100)]                  # (y, x) centres
template = zoo["4v4r"].sum(axis=0)                                      # 32 × 32 ribosome projection
for (py, px) in true_pos:
    field[py - 16:py + 16, px - 16:px + 16] += template
field_noisy = add_noise_snr(field, 0.3, rng)
tmpl_padded = np.zeros_like(field)
tmpl_padded[:32, :32] = template - template.mean()
tmpl_padded = np.roll(tmpl_padded, (-16, -16), axis=(0, 1))            # template centred on index (0, 0)
score = np.real(np.fft.ifft2(np.conj(np.fft.fft2(tmpl_padded)) * np.fft.fft2(field_noisy - field_noisy.mean())))
peaks = []
s_work = score.copy()
for _ in range(4):                                                      # take the 4 best peaks, suppressing neighbours
    p = np.unravel_index(np.argmax(s_work), s_work.shape)
    peaks.append(p)
    s_work[max(p[0] - 16, 0):p[0] + 16, max(p[1] - 16, 0):p[1] + 16] = -np.inf
print("found particle centres (y, x):", sorted(tuple(int(v) for v in p) for p in peaks))
print("true particle centres   (y, x):", sorted(true_pos))

fig, axes = plt.subplots(1, 2, figsize=(8.5, 4))
axes[0].imshow(field_noisy); axes[0].set_title("noisy field (SNR 0.3)", fontsize=9)
axes[1].imshow(score, cmap="viridis"); axes[1].set_title("correlation with the ribosome template", fontsize=9)
for (py, px) in peaks:
    axes[1].scatter(px, py, s=150, facecolors="none", edgecolors="tab:red", lw=1.5)
for ax in axes:
    ax.set_xlabel("x (nm)"); ax.set_ylabel("y (nm)")
plt.tight_layout(); plt.show()

---
## 9 · A translation changes only the phase

Shift a signal by $a$: every wave inside it is shifted by $a$, so every Fourier coefficient's **phase** changes, but
its **amplitude** does not (the **shift theorem**):

$$f(x - a)\ \longleftrightarrow\ F(k)\,e^{-2\pi i\,k a}\qquad\Rightarrow\qquad |\mathcal F[f(x - a)]| = |F(k)|.$$

*In words:* the Fourier **magnitudes** $|F|$ do not care where the molecule sits in the box — they are
**translation-invariant**. The phase difference between the two transforms is a linear ramp $-2\pi k a$ whose slope
encodes the shift.

Why is this a big deal for alignment? Aligning two subtomograms means finding a rotation **and** a translation —
6 numbers. Since $|F|$ ignores translation, we can search for the **rotation alone** by comparing magnitudes, and only
afterwards find the translation with one FFT cross-correlation (Section 8). Two smaller searches replace one huge one.
(Rotating an object rotates its Fourier transform by the same rotation, so rotations remain visible in $|F|$.)

In [ ]:
# ── 9.1 Verify: |F| is shift-invariant, the phase difference is a ramp ──
shift = (5, -9)
F0 = np.fft.fft2(ribo2d)
F1 = np.fft.fft2(np.roll(ribo2d, shift, axis=(0, 1)))                       # circular shift: exact theorem
F2 = np.fft.fft2(ndimage.shift(ribo2d, shift, order=1))                    # ordinary shift (zero fill, interpolation)
print(f"relative change of |F| — circular shift: {np.linalg.norm(np.abs(F1) - np.abs(F0)) / np.linalg.norm(F0):.1e}; "
      f"ordinary shift inside a padded box: {np.linalg.norm(np.abs(F2) - np.abs(F0)) / np.linalg.norm(F0):.1e}")
phase_diff = np.angle(F1 * np.conj(F0))
ky, kx = freq_grid(ribo2d.shape)
predicted = np.angle(np.exp(-2j * np.pi * (ky * shift[0] + kx * shift[1])))
strong = np.abs(F0) > 0.01 * np.abs(F0).max()                              # where the phase is well defined
print("phase difference equals −2π k·a (mod 2π):", np.allclose(np.exp(1j * phase_diff[strong]),
                                                                 np.exp(1j * predicted[strong]), atol=1e-6))

fig, axes = plt.subplots(1, 3, figsize=(9.5, 2.9))
axes[0].imshow(np.log10(np.abs(np.fft.fftshift(F0)) + 1e-3), cmap="viridis"); axes[0].set_title("log |F|, original", fontsize=9)
axes[1].imshow(np.log10(np.abs(np.fft.fftshift(F1)) + 1e-3), cmap="viridis"); axes[1].set_title(f"log |F|, shifted by {shift}", fontsize=9)
axes[2].imshow(np.fft.fftshift(phase_diff), cmap="twilight"); axes[2].set_title("phase difference: a ramp (stripes)", fontsize=9)
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

### 9.2 Finding a rotation without knowing the shift

A noisy copy of the ribosome image is rotated by an unknown angle **and** shifted by an unknown amount. We (1) rotate
the clean template through 0°…359° and correlate its Fourier magnitudes with those of the target (a band of low and
middle frequencies, where the signal is), ignoring the shift entirely; then (2) for the best angle, run one FFT
cross-correlation to get the shift.

⚠️ **One subtlety.** For any real image $|F(k)| = |F(-k)|$, so a 180° rotation leaves the magnitudes unchanged:
step 1 can only find the angle **modulo 180°**. Step 2 resolves it — we try both candidates and keep the one whose
translational correlation peak is higher.

In [ ]:
# ── 9.2 Two-step alignment: rotation from |F|, then translation from the FFT cross-correlation ──
template2d = projection_64("4v4r")
angle_true, shift_true = 137.0, (6.0, -4.0)                              # degrees; (dx, dy) in pixels
target2d = add_noise_snr(rotate_image(template2d, np.radians(angle_true), shift_xy=shift_true), 1.0, rng)
kmag = radial_frequency(target2d.shape)
band = (kmag > 0.01) & (kmag < 0.25)
A_target = np.abs(np.fft.fft2(target2d))[band]
test_angles = np.arange(0, 360, 1.0)
t0 = time.perf_counter()
rot_score = np.array([pearson(np.abs(np.fft.fft2(rotate_image(template2d, np.radians(a))))[band], A_target)
                      for a in test_angles])
best_a = test_angles[np.argmax(rot_score)]
candidates = []
for a in [best_a, (best_a + 180) % 360]:                                  # resolve the 180° ambiguity
    c_ = fft_xcorr(rotate_image(template2d, np.radians(a)), target2d)
    dy_, dx_ = peak_shift(c_)
    candidates.append((c_.max(), a, (dx_, dy_)))
score_best, angle_est, shift_est = max(candidates)
err = abs((angle_est - angle_true + 180) % 360 - 180)
print(f"step 1 (magnitudes only): best angles {best_a:.0f}° or {(best_a + 180) % 360:.0f}°  "
      f"[{time.perf_counter() - t0:.2f} s for 360 angles]")
print(f"step 2 (translation search): chose {angle_est:.0f}° with shift (dx, dy) = {shift_est}")
print(f"truth: {angle_true:.0f}°, shift {shift_true};  angle error {err:.0f}°")

fig, axes = plt.subplots(1, 3, figsize=(10.5, 2.9))
axes[0].plot(test_angles, rot_score, color="tab:purple")
axes[0].axvline(angle_true, color="k", ls="--", lw=1, label="true angle")
axes[0].axvline((angle_true + 180) % 360, color="0.5", ls=":", lw=1, label="true + 180°")
axes[0].set_xlabel("test rotation (°)"); axes[0].set_ylabel("corr. of |F|"); axes[0].legend(fontsize=7)
axes[0].set_title("step 1: rotation from magnitudes", fontsize=9)
axes[1].imshow(target2d); axes[1].set_title("target (rotated, shifted, noisy)", fontsize=9); axes[1].axis("off")
axes[2].imshow(target2d, cmap="gray")
axes[2].contour(rotate_image(template2d, np.radians(angle_est), shift_xy=shift_est), levels=[0.35], colors="tab:orange")
axes[2].set_title("template placed by the 2-step estimate", fontsize=9); axes[2].axis("off")
plt.tight_layout(); plt.show()
assert err < 10, "the two-step search should recover the angle within a few degrees"

🔍 **What just happened?** Without knowing the shift, Fourier magnitudes pinned down the rotation (up to the 180°
twin), and one cross-correlation then gave the shift and resolved the twin. Errors of a few degrees remain, from noise
and interpolation; real methods refine locally afterwards (Module 13).

### 📄 Paper connection — Fourier magnitudes in fast subtomogram alignment

**Paper:** Xu M, Beck M, Alber F (2012). *High-throughput subtomogram alignment and classification by Fourier space
constrained fast volumetric matching.* J Struct Biol 178(2):152–164.
[doi:10.1016/j.jsb.2012.02.014](https://doi.org/10.1016/j.jsb.2012.02.014) (open access: PMC3821800)

- **Question it answers:** aligning two noisy subtomograms needs a search over 3 rotation and 3 translation
  parameters, far too slow by brute force for reference-free classification of thousands of particles; and the missing
  wedge corrupts plain correlation.
- **What the paper actually does:** it compares the two volumes only on Fourier coefficients measured in *both* (a
  **constrained correlation**, Module 12). Because a translation only multiplies Fourier coefficients by a phase factor,
  it then drops the phases and scores rotations with **Fourier magnitudes**: a translation-invariant score
  $c'(R) = c_{R0}/\sqrt{c_{R1}c_{R2}}$ that is an upper bound of the full score over all translations. Each $c_{Ri}$ is a
  rotational correlation that is computed for **all rotations at once**, using spherical-harmonic expansions on
  concentric Fourier shells and one inverse FFT over Euler angles (the Kovacs & Wriggers 2002 approach). Fewer than 100
  local maxima of $c'$ are kept as candidate rotations, and an FFT translational search (our Section 8) picks the best
  rotation + translation.
- **What our toy keeps:** the separation of rotation (from translation-invariant magnitudes) and translation (from an
  FFT cross-correlation), and the use of a few candidates.
- **What our toy leaves out:** missing-wedge masks, the spherical-harmonic "all rotations at once" machinery (we loop
  over 360 angles in 2D), 3D, and classification. Module 12 implements the 2D "all angles at once" polar version.
- **Result reported:** the authors report that for 64³ volumes at 2.5° angular sampling their alignment took 153 s
  versus about 9.67 × 10⁵ s for exhaustive scanning (roughly 6000× faster), stayed accurate at SNR 0.01 and 0.005 for
  typical tilt ranges, and failed at 0.001.
- **Code:** AITom `aitom/align/fast/util.py` (`align_vols`, `fast_rotation_align`), calling the C++ core
  `aitom/tomominer/core/src/align.cpp`.

---
## 10 · Amplitude versus phase: which one carries the structure?

Each Fourier coefficient has an amplitude and a phase. Which matters more for what an image *looks like*? A classic
experiment: take the **amplitudes of image A** and the **phases of image B**, combine them, and transform back.

In [ ]:
# ── 10.1 Swap amplitudes and phases between two different images ──
c0_ = bgal_map.shape[0] // 2
img_A = bgal_map[c0_, c0_ - 32:c0_ + 32, c0_ - 32:c0_ + 32]                  # A: slice of the β-galactosidase map
img_A = (img_A - img_A.min()) / (img_A.max() - img_A.min())
img_B = np.zeros((64, 64), np.float32)                                      # B: three ribosomes in a small field
for (py, px) in [(18, 18), (20, 46), (46, 30)]:
    img_B[py - 16:py + 16, px - 16:px + 16] += zoo["4v4r"].sum(axis=0)
img_B /= img_B.max()
FA, FB = np.fft.fft2(img_A), np.fft.fft2(img_B)
hybrid_AB = np.real(np.fft.ifft2(np.abs(FA) * np.exp(1j * np.angle(FB))))     # amplitudes of A, phases of B
hybrid_BA = np.real(np.fft.ifft2(np.abs(FB) * np.exp(1j * np.angle(FA))))     # amplitudes of B, phases of A
print(f"A and B themselves: r = {pearson(img_A, img_B):.2f}")
print(f"hybrid (|A|, phase B): r = {pearson(hybrid_AB, img_B):.2f} with B, {pearson(hybrid_AB, img_A):.2f} with A")
print(f"hybrid (|B|, phase A): r = {pearson(hybrid_BA, img_A):.2f} with A, {pearson(hybrid_BA, img_B):.2f} with B")
fig, axes = plt.subplots(1, 4, figsize=(9.5, 2.7))
for ax, (im, t) in zip(axes, [(img_A, "A: β-galactosidase slice"), (img_B, "B: three ribosomes"),
                              (hybrid_AB, "amplitudes of A + phases of B"), (hybrid_BA, "amplitudes of B + phases of A")]):
    ax.imshow(im); ax.set_title(t, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each hybrid looks like (and correlates with) the image that supplied the **phases**. Phases encode *where*
features are; amplitudes mostly encode *how much* of each frequency. Two consequences you will meet again:

- 🧫 **The phase problem.** X-ray diffraction records only intensities $|F|^2$ — the phases are lost and must be
  recovered indirectly. Electron microscopes record *images*, which contain the phases: a key reason why cryo-EM can
  determine structures directly.
- **Magnitude-based scores are blind to some differences.** The rotational score of Section 9 compares magnitudes
  only — useful for orientation, but two different molecules can have similar magnitudes; that is why the final
  decision uses the full (phase-aware) correlation.

---
## 11 · Three dimensions: Fourier shells, radial spectra and the missing wedge

The 3D FFT (`np.fft.fftn`) works the same way: every coefficient has a 3D frequency vector $(k_z, k_y, k_x)$. Two facts
to check: the coefficient at $k = 0$ equals the sum of all voxels (the total "mass"); and frequencies with the same
length $|k|$ form a sphere — a **Fourier shell**. Averaging over shells gives 1D curves versus resolution, the basis of
the **Fourier shell correlation (FSC)** used to measure resolution (Module 10):

$$\mathrm{FSC}(r) = \frac{\sum_{k\in\text{shell } r} F_1(k)\,\overline{F_2(k)}}{\sqrt{\sum_{k\in r}|F_1(k)|^2\,\sum_{k\in r}|F_2(k)|^2}}$$

*In words:* the correlation between two independent maps, computed separately on each Fourier shell. Here we compute
the radially averaged **power spectrum** of a ribosome map and of noise, and the FSC between two independent noisy
copies (toolkit `fsc`).

In [ ]:
# ── 11.1 Radial power spectra and an FSC preview ──
ribo3d = zoo["4v4r"]
F3 = np.fft.fftn(ribo3d)
print(f"F(0,0,0) = {F3[0, 0, 0].real:.2f}  vs  sum of all voxels = {ribo3d.sum():.2f}")


def radial_average(values: np.ndarray, n_shells: int = 16) -> np.ndarray:
    """Average of `values` (unshifted FFT layout) over Fourier shells of width 1/(2·n_shells) cycles/voxel."""
    k = radial_frequency(values.shape)
    idx = np.minimum((k * 2 * n_shells).astype(int), n_shells)
    sums = np.bincount(idx.ravel(), values.ravel(), n_shells + 1)
    counts = np.bincount(idx.ravel(), minlength=n_shells + 1)
    return (sums / np.maximum(counts, 1))[:n_shells]


snr3d = 0.5
noise_sigma = np.sqrt(ribo3d.var() / snr3d)
copy1 = ribo3d + rng.normal(0, noise_sigma, ribo3d.shape)
copy2 = ribo3d + rng.normal(0, noise_sigma, ribo3d.shape)
p_signal = radial_average(np.abs(F3)**2)
p_noise = radial_average(np.abs(np.fft.fftn(rng.normal(0, noise_sigma, ribo3d.shape)))**2)
freq_A, fsc_curve, _ = fsc(copy1, copy2, pixel_A=10.0)
res_143 = resolution_at(freq_A, fsc_curve, 0.143)
below_floor = p_signal < p_noise
cross_txt = f"≈ {1 / freq_A[np.argmax(below_floor)]:.0f} Å" if below_floor.any() else "nowhere (beyond Nyquist)"
print(f"signal power falls below the noise floor at {cross_txt}; FSC(0.143) between the two noisy copies: "
      f"≈ {res_143:.0f} Å (voxel 10 Å, Nyquist 20 Å)")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.2))
axes[0].semilogy(freq_A, p_signal, "o-", label="ribosome (signal)")
axes[0].semilogy(freq_A, p_noise, "s-", label=f"white noise (SNR {snr3d})")
axes[0].set_xlabel("spatial frequency (1/Å)"); axes[0].set_ylabel("shell-averaged power |F|²")
axes[0].set_title("Radial power spectra"); axes[0].legend(fontsize=8)
axes[1].plot(freq_A, fsc_curve, "o-", color="tab:green")
axes[1].axhline(0.143, color="k", ls="--", lw=1); axes[1].text(0.002, 0.17, "0.143", fontsize=8)
axes[1].set_xlabel("spatial frequency (1/Å)"); axes[1].set_ylabel("FSC")
axes[1].set_title(f"FSC of two independent noisy copies (≈ {res_143:.0f} Å)")
for ax in axes:
    sec = ax.secondary_xaxis("top", functions=(lambda f: 1 / np.maximum(f, 1e-6), lambda d: 1 / np.maximum(d, 1e-6)))
    sec.set_xticks([200, 100, 50, 30, 20]); sec.set_xlabel("resolution (Å)", fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The molecule's power falls with frequency while white noise stays flat. Where the two are
equal, the FSC of two independent noisy copies is about 0.5 (for such copies FSC = S/(S + N), with S and N the signal and
noise power in the shell); it reaches the commonly used 0.143 threshold a little further out, where the signal is
already about 6× weaker than the noise (Module 10 explains the thresholds). Averaging more copies lowers the noise floor
and pushes both points to finer resolution (Modules 04, 10, 14).

**Missing-wedge preview.** In tomography the specimen can only be tilted to about ±60°, so a wedge of Fourier space is
never measured (Module 07). Applying the toolkit's `missing_wedge_mask` is just a Fourier filter — and its real-space
effect is an elongation of objects along the beam direction z.

In [ ]:
# ── 11.2 The missing wedge is a Fourier mask ──
wedge = missing_wedge_mask(ribo3d.shape, tilt_max_deg=60)
ribo_wedge = apply_fourier_mask(ribo3d, wedge)
print(f"fraction of Fourier coefficients measured with ±60° tilts: {wedge.mean():.0%}")
fig, axes = plt.subplots(1, 4, figsize=(10.5, 2.8))
logF = np.log10(np.abs(np.fft.fftshift(F3))[:, 16, :] + 1e-3)                # the k_y = 0 plane: (k_z, k_x)
axes[0].imshow(logF, cmap="viridis", extent=(-0.5, 0.5, 0.5, -0.5)); axes[0].set_title("log|F|, (k_z, k_x) plane", fontsize=8)
axes[1].imshow(np.where(np.fft.fftshift(wedge)[:, 16, :] > 0, logF, np.nan), cmap="viridis",
               extent=(-0.5, 0.5, 0.5, -0.5), vmin=logF.min(), vmax=logF.max())
axes[1].set_title("after the ±60° wedge mask", fontsize=8)
axes[2].imshow(ribo3d[:, 16, :]); axes[2].set_title("xz slice, full data", fontsize=8)
axes[3].imshow(ribo_wedge[:, 16, :]); axes[3].set_title("xz slice, missing wedge", fontsize=8)
for ax in axes[:2]:
    ax.set_xlabel("k_x"); ax.set_ylabel("k_z")
for ax in axes[2:]:
    ax.set_xlabel("x (nm)"); ax.set_ylabel("z (nm)")
plt.tight_layout(); plt.show()

---
## 12 · The discrete cosine transform and spectral pooling

Neural networks shrink feature maps step by step ("pooling"). The usual **max pooling** keeps the largest value in each
2×2 block — cheap, but it throws away *exactly where* inside the block a feature was. For **alignment**, positions are
the whole point. Gum-Net (Zeng & Xu 2020; Module 21) therefore downsamples in frequency space with the **discrete cosine
transform (DCT)**: transform, keep only the low-frequency block, transform back — **spectral pooling**.

🧮 **Why the cosine transform?** The DFT treats a signal as periodic, so the jump between its last and first sample
looks like a sharp edge and leaks energy into high frequencies. The DCT (type II) instead treats the signal as mirrored
at its ends — no jump — and uses only cosines. Its orthonormal 1D form is

$$X_k = \sqrt{\tfrac{2}{N}}\,\varepsilon_k \sum_{n=0}^{N-1} x_n \cos\!\Big(\frac{\pi k(2n+1)}{2N}\Big),\qquad \varepsilon_0 = \tfrac{1}{\sqrt2},\ \varepsilon_{k>0} = 1,$$

real-valued, invertible, and very good at packing a smooth signal's energy into a few low-$k$ coefficients (the same
reason JPEG uses it). Gum-Net's Eq. 1 is the 3D version (a product of three such cosines, with the factor
$\frac{8}{LWH}\varepsilon_l\varepsilon_h\varepsilon_w$ — the same transform up to a constant scale per axis); in SciPy it is
`scipy.fft.dctn(v, type=2, norm="ortho")`.

In [ ]:
# ── 12.1 Energy compaction: DCT versus DFT on a non-periodic profile ──
row = ribo2d[32, 20:52] + np.linspace(0, 0.6, 32)                # a line through the ribosome on a sloping background
E_dct = np.cumsum(np.sort(sfft.dct(row, norm="ortho")**2)[::-1]) / np.sum(row**2)
E_dft = np.cumsum(np.sort(np.abs(np.fft.fft(row, norm="ortho"))**2)[::-1]) / np.sum(row**2)
for n_keep in [2, 4, 8]:
    print(f"energy in the {n_keep} largest coefficients: DCT {E_dct[n_keep - 1]:.4f}, DFT {E_dft[n_keep - 1]:.4f}")


def spectral_pool(v: np.ndarray, keep: int, out: int | None = None) -> np.ndarray:
    """DCT spectral pooling (Gum-Net Algorithm 1): keep the lowest `keep` DCT coefficients per axis,
    zero-pad to `out` (default = keep), and invert. Intensities are rescaled so a constant stays constant."""
    out = out or keep
    C = sfft.dctn(v, type=2, norm="ortho")
    block = C[tuple(slice(0, keep) for _ in v.shape)]
    padded = np.zeros((out,) * v.ndim)
    padded[tuple(slice(0, keep) for _ in v.shape)] = block
    return sfft.idctn(padded, type=2, norm="ortho") * np.sqrt(out / v.shape[0]) ** v.ndim


def max_pool(v: np.ndarray, f: int = 2) -> np.ndarray:
    """Max over non-overlapping f×f (or f×f×f) blocks."""
    shp = sum(((n // f, f) for n in v.shape), ())
    return v.reshape(shp).max(axis=tuple(range(1, 2 * v.ndim, 2)))


def avg_pool(v: np.ndarray, f: int = 2) -> np.ndarray:
    """Mean over non-overlapping blocks (the binning of Module 01)."""
    shp = sum(((n // f, f) for n in v.shape), ())
    return v.reshape(shp).mean(axis=tuple(range(1, 2 * v.ndim, 2)))


demo = rng.normal(size=(30, 30, 30))
print("Gum-Net-style pooling of a 30³ feature map, keeping 22 coefficients, output 26³:",
      spectral_pool(demo, keep=22, out=26).shape)

**Does pooling remember positions?** We move a narrow 1D "feature" (a Gaussian bump of width 0.8 px) in steps of
0.05 px across a 32-pixel signal, pool it to 16 pixels with the three methods, and estimate the bump's position from
the pooled signal (its centre of mass, mapped back to original pixels). A perfect method traces the diagonal.

In [ ]:
# ── 12.2 Position preservation: max pooling vs average pooling vs spectral pooling ──
x32 = np.arange(32)
true_x = np.arange(12, 16.001, 0.05)
methods = {"max pooling": max_pool, "average pooling": avg_pool, "spectral pooling": lambda v: spectral_pool(v, keep=16)}
estimates = {name: [] for name in methods}
for x0 in true_x:
    bump = np.exp(-(x32 - x0)**2 / (2 * 0.8**2))
    for name, pool in methods.items():
        p = pool(bump)
        j = np.arange(p.size)
        estimates[name].append(2 * (p * j).sum() / p.sum() + 0.5)        # pooled pixel j sits at 2j + 0.5
for name, est in estimates.items():
    print(f"{name:17s}: largest position error {np.max(np.abs(np.array(est) - true_x)):.3f} px")

# 2D: how much of an image survives 2× pooling? (pool to 16², upsample back by zero-padded inverse DCT, compare)
img32 = ribo2d[16:48, 16:48]
recon_err = {}
pooled_imgs = {}
for name, pool in [("max pooling", max_pool), ("average pooling", avg_pool),
                   ("spectral pooling", lambda v: spectral_pool(v, keep=16))]:
    small = pool(img32)
    pooled_imgs[name] = small
    back = spectral_pool(small, keep=16, out=32)                           # identical upsampling for all methods
    recon_err[name] = np.linalg.norm(back - img32) / np.linalg.norm(img32)
    print(f"{name:17s}: relative error after pooling to 16² and upsampling back: {recon_err[name]:.3f}")

In [ ]:
# ── 12.3 Plots: positions and pooled images ──
fig = plt.figure(figsize=(11, 3.2))
ax = fig.add_subplot(1, 4, 1)
for (name, est), col in zip(estimates.items(), ["tab:red", "tab:blue", "tab:green"]):
    ax.plot(true_x, np.array(est) - true_x, color=col, label=name)
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("true position (px)"); ax.set_ylabel("estimated − true (px)"); ax.legend(fontsize=7)
ax.set_title("position error after 2× pooling", fontsize=9)
for k_, (name, small) in enumerate(pooled_imgs.items()):
    ax = fig.add_subplot(1, 4, k_ + 2)
    ax.imshow(small); ax.axis("off")
    ax.set_title(f"{name}, 16²\n(reconstruction error {recon_err[name]:.3f})", fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Max pooling makes the estimated position wobble as the bump crosses pixel boundaries — it
keeps "something bright is in this block" but forgets where. Average pooling wobbles too, a little less: it cannot tell
where inside a 2-pixel block the bump sits. Spectral pooling keeps the position almost exactly, and also preserves the
most image information (smallest reconstruction error), because it keeps *all* frequencies below the new Nyquist limit
and nothing above it. The errors look small for one smooth bump and one pooling step, but a network pools several times
(Gum-Net four times) — use the ✏️ below to see how they change for sharper features and larger pooling factors.
Position fidelity is exactly what an alignment network needs when it later compares feature positions in two
subtomograms.

✏️ **Your turn.** Make the feature narrower (`0.8` → `0.5`) or wider (`0.8` → `2.0`) in 12.2. When does max pooling
hurt most? Then pool by 4 instead of 2 (`keep=8`, and `2 *` → `4 *`, `+ 0.5` → `+ 1.5` in the position mapping;
`max_pool(v, 4)`).

### 📄 Paper connection — DCT spectral pooling in Gum-Net

**Paper:** Zeng X, Xu M (2020). *Gum-Net: Unsupervised Geometric Matching for Fast and Accurate 3D Subtomogram Image
Alignment and Averaging.* Proc. IEEE/CVF CVPR 2020, pp. 4073–4084.
[CVF open access](https://openaccess.thecvf.com/content_CVPR_2020/papers/Zeng_Gum-Net_Unsupervised_Geometric_Matching_for_Fast_and_Accurate_3D_Subtomogram_CVPR_2020_paper.pdf)

- **Question it answers:** can a neural network align two subtomograms (rotation + translation) quickly and without
  ground-truth transforms for training?
- **What the paper actually does (the part relevant here):** a Siamese network extracts 3D feature maps from both
  subtomograms. Because "the exact feature spatial location is critical" for matching, it downsamples with a
  **DCT-based spectral pooling and filtering** layer instead of max pooling (Eq. 1: the 3D type-II DCT; Algorithm 1:
  DCT → crop the low-frequency block → zero-pad → inverse DCT). In the released code, pooling after four convolution
  layers maps sizes 30 → 26, 24 → 18, 16 → 12, 10 → 8 while keeping 22, 15, 10 and 7 coefficients per axis. The rest
  of the network (feature correlation, transform regression, spatial transformer, missing-wedge imputation,
  unsupervised 1 − correlation loss) is the topic of Module 21.
- **What our toy keeps:** the transform, the crop-and-pad pooling rule (our `spectral_pool(v, keep, out)`), and the
  motivation (positions survive pooling).
- **What our toy leaves out:** the network, training and everything downstream of pooling.
- **Result reported:** the authors report that Gum-Net reduced alignment error by 40–50 % relative to classical
  baselines and was 70–110× faster (GPU vs a single CPU core); their ablations include max- and average-pooling variants.
- **Code:** AITom `aitom/align/deep/gum/models/layers/SpectralPooling.py` (Keras).

---
## ✅ Checkpoint

1. A map has 4 Å voxels. What is its Nyquist resolution, and which frequency (in cycles/voxel and in Å⁻¹) is that?
   <details><summary>Answer</summary>8 Å; 0.5 cycles/voxel = 0.5/4 = 0.125 Å⁻¹.</details>
2. In a power spectrum of a noisy micrograph, what do you expect at high frequencies, and why?
   <details><summary>Answer</summary>A flat floor: white noise has equal power at all frequencies, while the molecule's
   power decays with frequency, so noise dominates at high frequency.</details>
3. Why does a hard low-pass cutoff create ripples, and how do you avoid them?
   <details><summary>Answer</summary>A step in Fourier space corresponds to an oscillating kernel in real space (Gibbs
   ringing). Use a soft (smoothly decaying) filter edge.</details>
4. You shift a subtomogram by 3 voxels. What happens to the magnitudes and to the phases of its Fourier transform?
   <details><summary>Answer</summary>Magnitudes are unchanged; each phase changes by −2π k·a (a linear ramp in k). This
   is why rotations can be searched with magnitudes before the translation is known.</details>
5. Why can Fourier magnitudes determine an in-plane rotation only modulo 180°?
   <details><summary>Answer</summary>For real images |F(k)| = |F(−k)|, and a 180° rotation maps k to −k, so the
   magnitudes look identical; the phase-aware translational correlation resolves the ambiguity.</details>
6. What does spectral pooling keep that max pooling throws away?
   <details><summary>Answer</summary>All low-frequency information below the new Nyquist limit, including sub-pixel
   positions of features; max pooling keeps only the strongest response per block.</details>

## 🧠 Questions & discussion

1. The Fourier transform is a change of *description*, not of information. Why then are some operations (filtering,
   shifting, correlating) so much easier in Fourier space?
2. A colleague bins a tomogram 4× by taking every 4th voxel instead of averaging. What artefact do you expect, and why?
3. In X-ray crystallography only amplitudes are measured. Using the swap experiment, explain why "the phase problem"
   is a problem.
4. The magnitude-based rotational score is translation-invariant. What information does it lose, and when could that
   make two *different* molecules look alike?
5. Why does the missing wedge elongate objects along z rather than along x?

## 🚀 Bonus — going further

- **2D "all angles at once":** resample |F| on polar rings and compute the rotational correlation for all angles with
  one 1D FFT along the angle (the 2D analogue of Xu 2012's spherical-harmonic trick) — Module 12 builds it.
- **Phase correlation:** divide the cross-power spectrum by its magnitude before the inverse FFT
  (`F̄G/|F̄G|`) and compare peak sharpness with plain cross-correlation in noise.
- **3D spectral pooling:** apply `spectral_pool` to a 32³ zoo map (keep 16) and compare with `max_pool` in 3D slices.
- **Deep dives in this repository:** [frequency filtering](../../../../sci_img/tutorial/ComputationalSignalBase/FrequencyFiltering/)
  and [low-SNR noise processing](../../LowSNRNoiseProcessing/).
- **Read:** the methods section of Xu, Beck & Alber 2012 (open access, PMC3821800) — you now know every symbol in its
  Eq. 1–3.

## 📝 Lab note

```text
Module 03 — date:
Input: zoo maps 4V4R, 3J9I, 1BXN painted from PDB models (32³, 1 nm); β-galactosidase, PDB 5A1A (3.54 Å/voxel); synthetic profiles
Output: spectra, filtered images, shifts, a 2-step rotation+translation estimate, FSC preview, pooled images
Method: FFT/IFFT, fftfreq/fftshift, Fourier masks, convolution theorem, FFT cross-correlation, shift theorem,
        amplitude–phase swap, shell averages, DCT spectral pooling
Key parameter changed & effect (e.g. cutoff, SNR, pooling factor):
Failure mode (e.g. 180° ambiguity, ringing, aliasing, wrap-around):
Paper link: Xu, Beck & Alber 2012 (|F| translation invariance); Gum-Net 2020 (DCT spectral pooling)
```

## 📚 References

1. Xu M, Beck M, Alber F. High-throughput subtomogram alignment and classification by Fourier space constrained fast
   volumetric matching. *J Struct Biol* 178(2):152–164 (2012). doi:10.1016/j.jsb.2012.02.014
2. Zeng X, Xu M. Gum-Net: Unsupervised geometric matching for fast and accurate 3D subtomogram image alignment and
   averaging. *CVPR 2020*, pp. 4073–4084.
3. Kovacs JA, Wriggers W. Fast rotational matching. *Acta Cryst D* 58:1282–1286 (2002). doi:10.1107/S0907444902009794
4. Oppenheim AV, Lim JS. The importance of phase in signals. *Proc IEEE* 69(5):529–541 (1981). doi:10.1109/PROC.1981.12022
5. Rippel O, Snoek J, Adams RP. Spectral representations for convolutional neural networks. *NeurIPS 2015* (spectral
   pooling with the DFT).
6. Ahmed N, Natarajan T, Rao KR. Discrete cosine transform. *IEEE Trans Comput* C-23(1):90–93 (1974). doi:10.1109/T-C.1974.223784
7. Cooley JW, Tukey JW. An algorithm for the machine calculation of complex Fourier series. *Math Comput* 19:297–301 (1965).
   doi:10.1090/S0025-5718-1965-0178586-1

## ➡️ Next

**[04 · Noise, statistics and inference: how to find signal you cannot see](04_noise_statistics_inference.ipynb)** — a
single subtomogram has an SNR of 0.01–0.1. Next you will quantify noise, see why averaging N copies improves the signal
by √N, and learn the statistical tools (tests, mixture models, clustering) that turn noisy copies into biology.
Course overview: [`README.md`](../README.md) · terms: [`GLOSSARY.md`](../GLOSSARY.md).